# 05 - Entrenamiento VAE con datos reales

Cuaderno para entrenar y evaluar un Autoencoder Variacional (VAE) utilizando capturas reales de la maqueta de tren.

Convención actual de nomenclatura de datos:

- `real_movement_*.csv`: Capturas normales válidas para las fases de entrenamiento y validación.
- `real_anomaly_*.csv`: Capturas reales que presentan alteraciones mecánicas inducidas para la fase de evaluación.
- `debug_*` y `excluded_*`: Se excluyen automáticamente del procesamiento.

El modelo se entrena exclusivamente con datos correspondientes a un comportamiento normal. Las anomalías se detectan mediante la evaluación por ventanas, identificando aquellas cuyo score de reconstrucción (VAE score) supera un umbral previamente calculado sobre el conjunto de validación normal.


In [ ]:
from pathlib import Path
import random
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

plt.style.use("seaborn-v0_8-whitegrid")


: 

In [ ]:
def find_project_root():
    candidates = [
        Path.cwd().resolve(),
        Path.cwd().resolve().parent,
        Path.cwd().resolve().parent.parent,
        Path("/workspace/TFM"),
    ]
    for candidate in candidates:
        if (candidate / "datos" / "brutos").exists() and (candidate / "inference_server").exists():
            return candidate
    raise FileNotFoundError("No se encontró el directorio raíz del proyecto")

PROJECT_ROOT = find_project_root()
RAW_DIR = PROJECT_ROOT / "datos" / "brutos"
MODEL_DIR = PROJECT_ROOT / "inference_server" / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT={PROJECT_ROOT}")
print(f"RAW_DIR={RAW_DIR}")
print(f"MODEL_DIR={MODEL_DIR}")


## Selección de capturas

Este cuaderno emplea exclusivamente los siguientes conjuntos de datos:

```text
real_movement_*.csv -> Capturas normales reales válidas
real_anomaly_*.csv  -> Capturas con anomalías reales válidas
```

Los archivos etiquetados con prefijos `excluded_*` o `debug_*` se omiten en las etapas de entrenamiento y evaluación.


In [ ]:
normal_files = sorted(RAW_DIR.glob("real_movement_*.csv"))
anomaly_files = sorted(RAW_DIR.glob("real_anomaly_*.csv"))

print("Archivos normales para entrenamiento (Normal training files):")
for p in normal_files:
    print(" -", p.name)

print("\nArchivos con anomalías para evaluación (Anomaly evaluation files):")
for p in anomaly_files:
    print(" -", p.name)

if not normal_files:
    raise FileNotFoundError("No se han encontrado capturas real_movement_*.csv para realizar el entrenamiento")
if not anomaly_files:
    print("ADVERTENCIA: No se han encontrado capturas real_anomaly_*.csv; se omitirá la evaluación de anomalías")


In [ ]:
SEED = 42
SAMPLE_RATE_HZ = 100
WINDOW_SIZE = 100
WINDOW_STEP = 50
BATCH_SIZE = 512
EPOCHS = 30
LEARNING_RATE = 1e-3
LATENT_DIM = 16
BETA = 1e-3
THRESHOLD_PERCENTILE = 99.0
MAX_TRAIN_WINDOWS = 20000
VALIDATION_RATIO = 0.2
MAX_ABS_G = 8.0

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"DEVICE={DEVICE}")
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


In [ ]:
REQUIRED_COLUMNS = ["seq", "timestamp_ms", "acc_x_g", "acc_y_g", "acc_z_g"]
FEATURE_COLUMNS = ["acc_x_g", "acc_y_g", "acc_z_g"]


def read_capture(path):
    df = pd.read_csv(path)
    missing = [col for col in REQUIRED_COLUMNS if col not in df.columns]
    if missing:
        raise ValueError(f"{path.name} missing columns: {missing}")
    df = df.drop_duplicates(subset=["seq"], keep="first").sort_values("seq").reset_index(drop=True)
    return df


def quality_summary(df):
    seq = df["seq"].to_numpy(np.int64)
    ts = df["timestamp_ms"].to_numpy(np.int64)
    dseq = np.diff(seq)
    lost = int(np.maximum(dseq - 1, 0).sum())
    resets = int((dseq < 0).sum())
    duration_s = (ts[-1] - ts[0]) / 1000 if len(ts) > 1 else np.nan
    rate_hz = len(df) / duration_s if duration_s and duration_s > 0 else np.nan
    return {
        "rows": len(df),
        "seq_first": int(seq[0]),
        "seq_last": int(seq[-1]),
        "resets_after_sort": resets,
        "lost_packets": lost,
        "duration_s": duration_s,
        "rate_hz": rate_hz,
    }


def signal_summary(df):
    rows = {}
    abs_bad = (df[FEATURE_COLUMNS].abs() > MAX_ABS_G).any(axis=1)
    rows["rows_abs_over_8g"] = int(abs_bad.sum())
    rows["rows_abs_over_8g_ratio"] = float(abs_bad.mean())
    for col in FEATURE_COLUMNS:
        s = df[col]
        rows[f"{col}_min"] = float(s.min())
        rows[f"{col}_max"] = float(s.max())
        rows[f"{col}_abs_over_8g"] = int((s.abs() > MAX_ABS_G).sum())
    return rows

quality_rows = []
normal_dfs = []
for path in normal_files:
    df = read_capture(path)
    normal_dfs.append(df)
    quality_rows.append({"file": path.name, "type": "normal", **quality_summary(df), **signal_summary(df)})

anomaly_dfs = []
for path in anomaly_files:
    df = read_capture(path)
    anomaly_dfs.append(df)
    quality_rows.append({"file": path.name, "type": "anomaly", **quality_summary(df), **signal_summary(df)})

quality_df = pd.DataFrame(quality_rows)
display(quality_df)


In [ ]:
def make_windows_from_df(df, window_size=100, step=50, max_abs_g=8.0):
    values = df[FEATURE_COLUMNS].to_numpy(dtype=np.float32)
    if len(values) < window_size:
        return np.empty((0, window_size, 3), dtype=np.float32), np.empty((0,), dtype=np.int64)

    windows = []
    starts_kept = []
    starts = range(0, len(values) - window_size + 1, step)
    for start in starts:
        window = values[start:start + window_size]
        if np.isfinite(window).all() and np.max(np.abs(window)) <= max_abs_g:
            windows.append(window)
            starts_kept.append(start)

    if not windows:
        return np.empty((0, window_size, 3), dtype=np.float32), np.empty((0,), dtype=np.int64)
    return np.asarray(windows, dtype=np.float32), np.asarray(starts_kept, dtype=np.int64)

all_windows = []
window_sources = []
for path, df in zip(normal_files, normal_dfs):
    windows, starts = make_windows_from_df(df, WINDOW_SIZE, WINDOW_STEP, MAX_ABS_G)
    total_possible = max(0, (len(df) - WINDOW_SIZE) // WINDOW_STEP + 1)
    dropped = total_possible - len(windows)
    all_windows.append(windows)
    window_sources.extend([path.name] * len(windows))
    print(path.name, windows.shape, f"dropped_windows={dropped}")

windows_raw = np.concatenate(all_windows, axis=0)
window_sources = np.asarray(window_sources)
print("Total de ventanas normales procesadas:", windows_raw.shape)


In [ ]:
# División de datos por ventanas. Para realizar una evaluación más rigurosa, se podría implementar una división a nivel de archivo.
indices = np.random.permutation(len(windows_raw))
val_n = int(len(indices) * VALIDATION_RATIO)
val_idx = indices[:val_n]
train_idx = indices[val_n:]

if len(train_idx) > MAX_TRAIN_WINDOWS:
    train_idx = np.random.choice(train_idx, size=MAX_TRAIN_WINDOWS, replace=False)

train_raw = windows_raw[train_idx]
val_raw = windows_raw[val_idx]
print("train_raw", train_raw.shape)
print("val_raw", val_raw.shape)


In [ ]:
def fit_standardizer(windows):
    flat = windows.reshape(-1, windows.shape[-1])
    mean = flat.mean(axis=0).astype(np.float32)
    std = flat.std(axis=0).astype(np.float32)
    std = np.where(std < 1e-6, 1.0, std).astype(np.float32)
    return {"mean": mean, "std": std}


def transform_standardizer(windows, scaler):
    return ((windows - scaler["mean"]) / scaler["std"]).astype(np.float32)


def to_channels_first(windows):
    return np.transpose(windows, (0, 2, 1)).astype(np.float32)

scaler = fit_standardizer(train_raw)
train_x = to_channels_first(transform_standardizer(train_raw, scaler))
val_x = to_channels_first(transform_standardizer(val_raw, scaler))
print("mean", scaler["mean"])
print("std", scaler["std"])


In [ ]:
class ConvVAE(nn.Module):
    def __init__(self, in_channels=3, window_size=100, latent_dim=16):
        super().__init__()
        self.window_size = window_size
        self.encoder = nn.Sequential(
            nn.Conv1d(in_channels, 32, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Conv1d(32, 64, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Conv1d(64, 128, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
        )
        with torch.no_grad():
            encoded = self.encoder(torch.zeros(1, in_channels, window_size))
        self.encoded_shape = tuple(encoded.shape[1:])
        flat_dim = int(np.prod(self.encoded_shape))
        self.fc_mu = nn.Linear(flat_dim, latent_dim)
        self.fc_logvar = nn.Linear(flat_dim, latent_dim)
        self.fc_decode = nn.Linear(latent_dim, flat_dim)
        self.decoder = nn.Sequential(
            nn.ConvTranspose1d(128, 64, kernel_size=5, stride=2, padding=2, output_padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.ConvTranspose1d(64, 32, kernel_size=5, stride=2, padding=2, output_padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.ConvTranspose1d(32, in_channels, kernel_size=7, stride=2, padding=3, output_padding=1),
        )

    def encode(self, x):
        h = self.encoder(x).flatten(1)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + torch.randn_like(std) * std

    def decode(self, z):
        h = self.fc_decode(z).view(-1, *self.encoded_shape)
        out = self.decoder(h)
        if out.size(-1) != self.window_size:
            out = F.interpolate(out, size=self.window_size, mode="linear", align_corners=False)
        return out

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar


def vae_loss(x, reconstruction, mu, logvar, beta):
    recon = F.mse_loss(reconstruction, x, reduction="mean")
    kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    return recon + beta * kl, recon, kl


In [ ]:
def loader(x, shuffle=False):
    return DataLoader(TensorDataset(torch.from_numpy(x)), batch_size=BATCH_SIZE, shuffle=shuffle)

model = ConvVAE(window_size=WINDOW_SIZE, latent_dim=LATENT_DIM).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
train_loader = loader(train_x, shuffle=True)
val_loader = loader(val_x, shuffle=False)
print(f"parameters={sum(p.numel() for p in model.parameters()):,}")


In [ ]:
def run_epoch(data_loader, train=True):
    model.train(train)
    totals = {"loss": 0.0, "recon": 0.0, "kl": 0.0, "n": 0}
    for (batch,) in data_loader:
        batch = batch.to(DEVICE)
        if train:
            optimizer.zero_grad(set_to_none=True)
        reconstruction, mu, logvar = model(batch)
        loss, recon, kl = vae_loss(batch, reconstruction, mu, logvar, BETA)
        if train:
            loss.backward()
            optimizer.step()
        bs = batch.size(0)
        totals["loss"] += float(loss.detach().cpu()) * bs
        totals["recon"] += float(recon.detach().cpu()) * bs
        totals["kl"] += float(kl.detach().cpu()) * bs
        totals["n"] += bs
    return {k: totals[k] / totals["n"] for k in ["loss", "recon", "kl"]}

history = []
start = time.perf_counter()
for epoch in range(1, EPOCHS + 1):
    train_metrics = run_epoch(train_loader, train=True)
    with torch.no_grad():
        val_metrics = run_epoch(val_loader, train=False)
    row = {"epoch": epoch, **{f"train_{k}": v for k, v in train_metrics.items()}, **{f"val_{k}": v for k, v in val_metrics.items()}}
    history.append(row)
    print(f"epoch={epoch:03d} train_loss={row['train_loss']:.6f} val_loss={row['val_loss']:.6f} val_recon={row['val_recon']:.6f} val_kl={row['val_kl']:.6f}")
print(f"Tiempo de entrenamiento (segundos)={time.perf_counter() - start:.2f}")


In [ ]:
history_df = pd.DataFrame(history)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(history_df["epoch"], history_df["train_loss"], label="train")
axes[0].plot(history_df["epoch"], history_df["val_loss"], label="val")
axes[0].set_title("Pérdida (Loss) Total")
axes[0].legend()
axes[1].plot(history_df["epoch"], history_df["val_recon"], label="reconstruction")
axes[1].plot(history_df["epoch"], history_df["val_kl"], label="KL")
axes[1].set_title("Términos de Validación")
axes[1].legend()
plt.show()


In [ ]:
@torch.no_grad()
def score_windows(x):
    model.eval()
    rows = []
    for (batch,) in loader(x, shuffle=False):
        batch = batch.to(DEVICE)
        reconstruction, mu, logvar = model(batch)
        recon = torch.mean((batch - reconstruction) ** 2, dim=(1, 2))
        kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp(), dim=1)
        score = recon + BETA * kl
        rows.append(torch.stack([recon, kl, score], dim=1).cpu().numpy())
    arr = np.concatenate(rows, axis=0)
    return pd.DataFrame(arr, columns=["reconstruction_error", "kl", "vae_score"])

val_scores = score_windows(val_x)
threshold = float(np.percentile(val_scores["vae_score"], THRESHOLD_PERCENTILE))
print(f"threshold p{THRESHOLD_PERCENTILE}={threshold:.6f}")
display(val_scores.describe())


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, ["reconstruction_error", "kl", "vae_score"]):
    ax.hist(val_scores[col], bins=100)
    if col == "vae_score":
        ax.axvline(threshold, color="red", linestyle="--", label="threshold")
        ax.legend()
    ax.set_title(col)
plt.tight_layout()
plt.show()


## Evaluación frente a capturas anómalas

En esta etapa se aplica el mismo preprocesamiento y se utiliza el umbral de detección (threshold) previamente aprendido durante la validación sobre datos normales.


In [ ]:
def evaluate_capture(path, df, label):
    windows, starts = make_windows_from_df(df, WINDOW_SIZE, WINDOW_STEP, MAX_ABS_G)
    if len(windows) == 0:
        return None, None
    x = to_channels_first(transform_standardizer(windows, scaler))
    scores = score_windows(x)
    scores["window_start"] = starts
    scores["time_s"] = starts / SAMPLE_RATE_HZ
    scores["is_anomaly"] = scores["vae_score"] > threshold
    summary = {
        "file": path.name,
        "label": label,
        "windows": len(scores),
        "threshold": threshold,
        "mean_score": float(scores["vae_score"].mean()),
        "median_score": float(scores["vae_score"].median()),
        "max_score": float(scores["vae_score"].max()),
        "anomaly_windows": int(scores["is_anomaly"].sum()),
        "anomaly_ratio": float(scores["is_anomaly"].mean()),
    }
    return summary, scores

results = []
score_tables = {}

for path, df in zip(normal_files, normal_dfs):
    summary, scores = evaluate_capture(path, df, "normal")
    if summary:
        results.append(summary)
        score_tables[path.name] = scores

for path, df in zip(anomaly_files, anomaly_dfs):
    summary, scores = evaluate_capture(path, df, "anomaly")
    if summary:
        results.append(summary)
        score_tables[path.name] = scores

results_df = pd.DataFrame(results)
display(results_df.sort_values(["label", "file"]))

fig, ax = plt.subplots(figsize=(14, 4))
for name, scores in score_tables.items():
    ax.plot(scores["time_s"], scores["vae_score"], label=name, alpha=0.8)
ax.axhline(threshold, color="red", linestyle="--", label="threshold")
ax.set_title("VAE Score por ventana")
ax.set_xlabel("time (s)")
ax.set_ylabel("vae_score")
ax.legend()
plt.show()


## Análisis temporal detallado de la captura anómala

El gráfico global presenta conjuntamente un periodo extenso de datos normales y una captura anómala de menor duración. En esta vista se realiza un zoom temporal sobre el primer minuto de la captura anómala para inspeccionar con mayor precisión los picos de los scores obtenidos por ventana.


In [ ]:
ZOOM_FILE = "real_anomaly_001.csv"
ZOOM_START_S = 0
ZOOM_DURATION_S = 60

if ZOOM_FILE not in score_tables:
    raise KeyError(f"{ZOOM_FILE} no se encuentra en score_tables. Verifique que exista como real_anomaly_*.csv")

zoom_scores = score_tables[ZOOM_FILE].copy()
zoom_end_s = ZOOM_START_S + ZOOM_DURATION_S
zoom_scores = zoom_scores[(zoom_scores["time_s"] >= ZOOM_START_S) & (zoom_scores["time_s"] <= zoom_end_s)]

print(f"{ZOOM_FILE}: ventanas incluidas en el zoom={len(zoom_scores)}")
print(f"proporción de ventanas anómalas en el zoom={zoom_scores['is_anomaly'].mean():.3%}")
print(f"score máximo detectado en el zoom={zoom_scores['vae_score'].max():.6f}")

fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(zoom_scores["time_s"], zoom_scores["vae_score"], marker="o", markersize=3, linewidth=1.4, label="VAE score")
ax.axhline(threshold, color="red", linestyle="--", label=f"threshold p{THRESHOLD_PERCENTILE:g}")

anomaly_points = zoom_scores[zoom_scores["is_anomaly"]]
ax.scatter(anomaly_points["time_s"], anomaly_points["vae_score"], color="red", s=28, label="ventana clasificada como anómala")

ax.set_title(f"Análisis detallado VAE Score - {ZOOM_FILE} ({ZOOM_START_S}-{zoom_end_s}s)")
ax.set_xlabel("time (s)")
ax.set_ylabel("vae_score")
ax.legend()
plt.show()



In [ ]:
# Guardado opcional del modelo entrenado. Se recomienda establecer SAVE_MODEL=True una vez que el entrenamiento converja de forma satisfactoria.
SAVE_MODEL = False
artifact_path = MODEL_DIR / "vae_real_v1.pth"

if SAVE_MODEL:
    artifact = {
        "model_state_dict": model.state_dict(),
        "config": {
            "model_type": "ConvVAE",
            "window_size": WINDOW_SIZE,
            "window_step": WINDOW_STEP,
            "sample_rate_hz": SAMPLE_RATE_HZ,
            "latent_dim": LATENT_DIM,
            "beta": BETA,
            "feature_columns": FEATURE_COLUMNS,
            "training_files": [p.name for p in normal_files],
            "note": "Trained on valid real_movement captures. Windows with abs(acc)>8g were discarded as sensor/I2C glitches.",
        },
        "normalization": {
            "mean": scaler["mean"].tolist(),
            "std": scaler["std"].tolist(),
        },
        "threshold": threshold,
        "threshold_percentile": THRESHOLD_PERCENTILE,
    }
    torch.save(artifact, artifact_path)
    print(f"Modelo guardado exitosamente en {artifact_path}")
else:
    print("SAVE_MODEL es False; el modelo no se ha guardado")


## Siguientes pasos

Una vez que el entrenamiento converge y las distribuciones de scores resultan coherentes, se deben considerar las siguientes acciones:

1. Recolectar nuevas capturas `real_movement_*.csv` utilizando el firmware configurado a `±8g`.
2. Ejecutar un nuevo entrenamiento para obtener un modelo base exento de saturaciones relevantes en la señal.
3. Recolectar capturas experimentales de la vía con alteraciones estructurales empleando una nueva convención de nomenclatura que se definirá para dicha fase.
4. Realizar la evaluación exhaustiva del modelo VAE utilizando datos reales, contrastando su rendimiento frente a anomalías y frente a modelos de referencia (baseline).
